# Day 2 — Chunking

Goal: take the documents from Day 1 and cut them into search-sized pieces.
Some dbt docs are tiny; the largest is ~122,000 characters. That spread is the
whole reason chunking exists — one document is 25× the median.

My approach evolved past the course's basic sliding window into a cascade:
**split on headers first**, fall back to sliding window only for oversized
sections. Headers are split points the docs' authors already marked, so this
keeps topics intact and avoids cutting through code blocks.

In [1]:
import io
import json
import os
import re
import statistics
import zipfile

import frontmatter
import requests

OWNER = "dbt-labs"
REPO = "docs.getdbt.com"
BRANCH = "current"
ZIP_CACHE = "repo.zip"

## 1. Load the documents

Same ingestion as Day 1, but I cache the 550 MB zip to disk. Without this,
every notebook restart re-downloads it — and on Day 2 I re-run constantly while
tuning chunk sizes.

In [2]:
def download_repo(owner, repo, branch, cache_path):
    if not os.path.exists(cache_path):
        url = f"https://codeload.github.com/{owner}/{repo}/zip/refs/heads/{branch}"
        print(f"downloading {url}")
        resp = requests.get(url, timeout=300)
        resp.raise_for_status()
        with open(cache_path, "wb") as f:
            f.write(resp.content)
        print(f"cached {len(resp.content) / 1_000_000:.1f} MB")
    return zipfile.ZipFile(cache_path)


def read_repo_data(zf):
    documents = []
    for info in zf.infolist():
        if not info.filename.lower().endswith((".md", ".mdx")):
            continue
        with zf.open(info) as f:
            post = frontmatter.loads(f.read().decode("utf-8", errors="ignore"))
        data = post.to_dict()
        data["filename"] = info.filename.split("/", 1)[1]   # strip zip root folder
        documents.append(data)
    return documents


zf = download_repo(OWNER, REPO, BRANCH, ZIP_CACHE)
dbt_labs = read_repo_data(zf)
print(f"documents: {len(dbt_labs)}")

documents: 1547


## 2. The splitters

### Sliding window (the course's baseline)
Fixed-size overlapping windows. Simple, but it cuts blindly — straight through
sentences and code blocks. I keep it as a last-resort fallback.

In [3]:
def sliding_window(seq, size, step):
    """Fixed-size overlapping windows. The break prevents trailing duplicates."""
    if size <= 0 or step <= 0:
        raise ValueError("size and step must be positive")
    n = len(seq)
    result = []
    for i in range(0, n, step):
        result.append({"start": i, "chunk": seq[i:i + size]})
        if i + size >= n:
            break
    return result

### Header splitter
Splits on markdown headers (`##`). **Key fix vs. the naive version:** it keeps
`parts[0]` — the preamble before the first header. That preamble is ~19% of the
corpus and usually the clearest summary of the page. The naive loop starts at
index 1 and silently throws it away.

In [4]:
def split_markdown_by_level(text, level=2):
    """Split on markdown headers, KEEPING the preamble before the first header."""
    pattern = re.compile(r"^(#{" + str(level) + r"}\s+.+)$", re.MULTILINE)
    parts = pattern.split(text)

    sections = []
    if parts[0].strip():                     # 19% of the corpus lives here
        sections.append(parts[0].strip())

    for i in range(1, len(parts), 2):
        header = parts[i].strip()
        content = parts[i + 1].strip() if i + 1 < len(parts) else ""
        sections.append(header + "\n\n" + content)
    return sections

## 3. Chunk: sections first, sliding window only for oversized ones

The cascade for a too-long section: try `###` sub-headers, then paragraph
boundaries (tracking ``` fences so I never cut a code block in half), and only
blind-window a genuinely unsplittable blob. This took broken code fences from
~1,630 (pure sliding window) down to ~130.

In [5]:
MAX_CHARS = 4000      # anything longer gets windowed
WINDOW = 2000
STEP = 1000


def split_oversized(section, max_chars):
    """
    Break a too-long section without cutting through code blocks:
    try ### headers, then blank lines, and only window as a last resort.
    """
    if len(section) <= max_chars:
        return [section]

    # try the next header level down
    subs = split_markdown_by_level(section, level=3)
    if len(subs) > 1:
        return [p for s in subs for p in split_oversized(s, max_chars)]

    # accumulate paragraphs, never splitting inside a ``` fence
    pieces, buf, in_fence = [], [], False
    for para in re.split(r"\n\s*\n", section):
        if buf and not in_fence and len("\n\n".join(buf)) + len(para) > max_chars:
            pieces.append("\n\n".join(buf))
            buf = []
        buf.append(para)
        if para.count("```") % 2 == 1:
            in_fence = not in_fence
    if buf:
        pieces.append("\n\n".join(buf))

    # anything still oversized is one unsplittable blob - window it
    out = []
    for p in pieces:
        if len(p) <= max_chars:
            out.append(p)
        else:
            out.extend(w["chunk"] for w in sliding_window(p, WINDOW, STEP))
    return out

### Assemble chunks per document
Two things happen here that matter for Day 3:
1. **Title prepended into the chunk TEXT**, not just metadata — Day 3 searches
   the `chunk` string, so metadata in other keys is invisible to search.
2. **Filename fallback** for docs with no frontmatter title.

In [6]:
def chunk_document(doc, max_chars=MAX_CHARS):
    meta = doc.copy()
    content = meta.pop("content")

    # fall back to the filename when there is no frontmatter title
    title = str(meta.get("title") or "")
    if not title:
        title = meta["filename"].rsplit("/", 1)[-1].rsplit(".", 1)[0].replace("-", " ")
    description = str(meta.get("description") or "")
    header = "\n".join(x for x in (title, description) if x)

    chunks = []
    for section in split_markdown_by_level(content, level=2):
        for piece in split_oversized(section, max_chars):
            record = dict(meta)
            record["chunk"] = f"{header}\n\n{piece}".strip() if header else piece
            chunks.append(record)
    return chunks


dbt_chunks = []
for doc in dbt_labs:
    dbt_chunks.extend(chunk_document(doc))

print(f"chunks: {len(dbt_chunks)}")

chunks: 7910


## 4. Check the result

Compare against a pure sliding window: that produced ~9,300 uniform 2000-char
chunks with ~1,630 broken code fences. This produces fewer chunks, sized where
topics actually end, with far fewer broken fences.

In [7]:
lengths = [len(c["chunk"]) for c in dbt_chunks]
print(f"median size   : {int(statistics.median(lengths))}")
print(f"min / max     : {min(lengths)} / {max(lengths)}")
print(f"under 200 char: {sum(1 for n in lengths if n < 200)}")
print(f"over 4000 char: {sum(1 for n in lengths if n > 4000)}")
print(f"broken code   : {sum(1 for c in dbt_chunks if c['chunk'].count('```') % 2)}")
print(f"missing title : {sum(1 for c in dbt_chunks if not c.get('title'))}  (filename used as fallback)")

median size   : 1091
min / max     : 20 / 4498
under 200 char: 407
over 4000 char: 120
broken code   : 132
missing title : 881  (filename used as fallback)


## 5. Read some actual chunks
Numbers hide problems the eye catches instantly — always read a few.

In [8]:
for c in dbt_chunks[500:503]:
    print("=" * 70)
    print(c["filename"])
    print("-" * 70)
    print(c["chunk"][:400])
    print()

website/blog/2022-08-22-unit-testing-dbt-package.md
----------------------------------------------------------------------
An introduction to unit testing your dbt Packages
Traditionally, integration tests have been the primary strategy for testing dbt Packages. In this post, Yu Ishikawa walks us through adding in unit testing as well.

_Editors note - this post assumes working knowledge of dbt Package development. For an introduction to dbt Packages check out [So You Want to Build a dbt Package](https://docs.getdbt.c

website/blog/2022-08-22-unit-testing-dbt-package.md
----------------------------------------------------------------------
An introduction to unit testing your dbt Packages
Traditionally, integration tests have been the primary strategy for testing dbt Packages. In this post, Yu Ishikawa walks us through adding in unit testing as well.

## Unit Testing vs. Integration Testing

Unit testing and integration testing are two common paradigms in create well-tested code. For a

## 6. Save for Day 3
`chunks.jsonl` is the handoff file — Day 3 loads it to build the search index.

In [9]:
with open("chunks.jsonl", "w", encoding="utf-8") as out:
    for c in dbt_chunks:
        out.write(json.dumps(c, default=str, ensure_ascii=False) + "\n")
print(f"wrote chunks.jsonl ({len(dbt_chunks)} chunks)")

wrote chunks.jsonl (7910 chunks)


## 7. Homework check — wizard chunks

Count how many chunks mention Wizard, and eyeball the first few. Note the top
hits here are file-order, not relevance-order — fixing that is Day 3's job.

In [10]:
wizard_chunks = [c for c in dbt_chunks if "wizard" in c["chunk"].lower()]

print(f"total   : {len(dbt_chunks)}")
print(f"wizard  : {len(wizard_chunks)}")
print(f"files   : {len({c['filename'] for c in wizard_chunks})}")

for i, c in enumerate(wizard_chunks[:5], 1):
    print(f"\n{'='*70}\n[{i}] {c['filename']}\n{'-'*70}")
    print(c["chunk"][:450])

total   : 7910
wizard  : 420
files   : 109

[1] website/blog/2022-11-22-move-spreadsheets-to-your-dwh.md
----------------------------------------------------------------------
How to move data from spreadsheets into your data warehouse
A thankless, humble, and inevitable task: getting spreadsheet data into your data warehouse. Let's look at some of the different options, and the pros and cons of each.

## Native warehouse integrations

Each of the major data warehouses also has native integrations to import spreadsheet data. While the fundamentals are the same, there are some differences amongst the various warehousing

[2] website/blog/2025-04-10-dbt-cloud-sso-rbac.md
----------------------------------------------------------------------
Establishing dbt Cloud: Securing your account through SSO & RBAC
How to configure dbt Cloud with SSO & RBAC

As a dbt Cloud admin, you’ve just upgraded to dbt Cloud on the [Enterprise plan](https://www.getdbt.com/pricing) - **congrats**! dbt Cloud has

### Crude relevance ranking (a preview of Day 3)
Ranking by keyword density — how often "wizard" appears relative to chunk
length — is a poor man's TF-IDF. Day 3 does this properly with minsearch.

In [11]:
ranked = sorted(
    wizard_chunks,
    key=lambda c: c["chunk"].lower().count("wizard") / len(c["chunk"]),
    reverse=True,
)
[c["filename"].split("/")[-1] for c in ranked[:5]]

['wizard-migrate.md',
 'wizard-how-it-works.md',
 'wizard-use-cases.md',
 'wizard-skills.md',
 'wizard-platform-subagents.md']

### The real Wizard product docs
Filtering by path isolates the actual product documentation from pages that
merely mention Wizard in passing.

In [12]:
actual_wizard = [c for c in wizard_chunks if c["filename"].startswith("website/docs/docs/dbt-ai/")]
print(f"wizard product-doc chunks: {len(actual_wizard)}")
[c["filename"].split("/")[-1] for c in actual_wizard[:5]]

wizard product-doc chunks: 195


['_wizard-cli-full-generated.md',
 '_wizard-cli-full-generated.md',
 '_wizard-cli-full-generated.md',
 '_wizard-cli-full-generated.md',
 '_wizard-cli-full-generated.md']